# 台灣好聲音（PitchLab）雲端 AI 分軌與歌詞生成器
### 操作只要 2 步（自動存進 `麻糬紅豆湯/PitchLab_Songs`，App 自動同步）：
1. 上方選單 **「執行階段」→「變更執行階段類型」→ 選 `T4 GPU`**。
2. 點擊下方 **「步驟 1」左側的 ▶ 播放鈕**：授權雲端硬碟並安裝完成後，下方會**自動出現「選擇音樂檔」與「開始 AI 分軌」按鈕**，跑完會**自動存進 Google Drive 的 `麻糬紅豆湯/PitchLab_Songs/` 資料夾**（同時提供下載），打開網頁 App 就會自動匯入！

In [ ]:
#@title 步驟 1：連線雲端硬碟、安裝環境並顯示上傳面板（點左側 ▶ 執行）
from google.colab import drive, files
from pathlib import Path
import subprocess
import ipywidgets as widgets
from IPython.display import display, clear_output

drive.mount('/content/drive')
drive_out_dir = Path('/content/drive/MyDrive/麻糬紅豆湯/PitchLab_Songs')
drive_out_dir.mkdir(parents=True, exist_ok=True)

!git clone https://github.com/bard0228/pitchlab.git /content/pitchlab 2>/dev/null || (cd /content/pitchlab && git pull)
!pip -q install -r /content/pitchlab/pipeline/requirements.txt
!apt-get -qq install -y ffmpeg

uploader = widgets.FileUpload(accept='.mp3,.wav,.m4a,.flac,.ogg,.aac', multiple=False, description='選擇音樂檔')
title_box = widgets.Text(value='', placeholder='留空則自動使用檔名', description='歌名：')
lyrics_dd = widgets.Dropdown(
    options=[('自動辨識歌詞 (Whisper)', 'whisper'), ('不產生歌詞', 'none')],
    value='whisper',
    description='歌詞：',
)
run_btn = widgets.Button(description='開始 AI 分軌並存到雲端硬碟', button_style='success', icon='play')
status_out = widgets.Output()

def _extract_uploaded(val):
    if isinstance(val, dict) and val:
        name, info = next(iter(val.items()))
        return name, bytes(info['content'])
    if isinstance(val, (tuple, list)) and len(val) > 0:
        info = val[0]
        return info['name'], bytes(info['content'])
    return None, None

def _on_run(_):
    with status_out:
        clear_output()
        fname, fbytes = _extract_uploaded(uploader.value)
        if not fname:
            print('⚠️ 請先點擊上方「選擇音樂檔」上傳 MP3 / WAV / M4A！')
            return
        in_dir = Path('/content/pitchlab_in')
        in_dir.mkdir(exist_ok=True)
        src_path = in_dir / fname
        src_path.write_bytes(fbytes)
        song_title = title_box.value.strip() or src_path.stem
        out_zip = drive_out_dir / f'{song_title}.zip'
        cmd = [
            'python3', '/content/pitchlab/pipeline/process_song.py',
            '--input', str(src_path),
            '--separator', 'demucs',
            '--title', song_title,
            '--out', str(out_zip),
        ]
        if lyrics_dd.value == 'whisper':
            cmd += ['--lyrics-mode', 'whisper']
        print(f'🎵 正在處理《{song_title}》（Demucs 分軌 + {lyrics_dd.value}），請稍候...')
        res = subprocess.run(cmd, check=False)
        if res.returncode == 0 and out_zip.exists():
            print('✅ 完成！已自動存入雲端硬碟：麻糬紅豆湯/PitchLab_Songs/' + out_zip.name)
            print('📱 打開手機「台灣好聲音」App 就會自動匯入這首歌！')
        else:
            print('❌ 處理失敗，請檢查上方訊息。')

run_btn.on_click(_on_run)
print(f'✅ 環境與雲端硬碟就緒（輸出資料夾：{drive_out_dir}）！')
display(widgets.VBox([uploader, title_box, lyrics_dd, run_btn, status_out]))

In [ ]:
#@title 步驟 2（進階選用）：貼 YouTube 網址、純文字歌詞對齊或上傳 .lrc
YOUTUBE_URL = "" #@param {type:"string"}
SONG_TITLE = "" #@param {type:"string"}
LYRICS_MODE = "whisper" #@param ["whisper", "align_text", "upload_lrc", "none"]
PLAIN_LYRICS = "" #@param {type:"string"}

from google.colab import drive, files
from pathlib import Path
import subprocess

drive_out_dir = Path('/content/drive/MyDrive/麻糬紅豆湯/PitchLab_Songs')
if not drive_out_dir.parent.parent.exists():
    drive.mount('/content/drive')
drive_out_dir.mkdir(parents=True, exist_ok=True)
tmp_dir = Path('/content/pitchlab_tmp')
tmp_dir.mkdir(exist_ok=True)

cmd = ['python3', '/content/pitchlab/pipeline/process_song.py', '--separator', 'demucs']
if YOUTUBE_URL.strip():
    cmd += ['--youtube', YOUTUBE_URL.strip()]
else:
    print('👇 請點下方「選擇檔案」按鈕上傳 MP3 / WAV / M4A：')
    uploaded = files.upload()
    src_file = next(iter(uploaded.keys()))
    cmd += ['--input', src_file]
    if not SONG_TITLE.strip():
        SONG_TITLE = Path(src_file).stem

if LYRICS_MODE == 'upload_lrc':
    print('👇 請上傳對應的 .lrc 動態歌詞檔：')
    lrc_up = files.upload()
    lrc_file = next(iter(lrc_up.keys()))
    cmd += ['--lrc', lrc_file]
elif LYRICS_MODE == 'align_text':
    txt_path = tmp_dir / 'plain_lyrics.txt'
    if PLAIN_LYRICS.strip():
        txt_path.write_text(PLAIN_LYRICS.replace('\\n', '\n'), encoding='utf-8')
    else:
        print('👇 請上傳純文字歌詞 .txt 檔：')
        txt_up = files.upload()
        txt_path = Path(next(iter(txt_up.keys())))
    cmd += ['--lyrics-mode', 'align', '--lyrics-text', str(txt_path)]
elif LYRICS_MODE == 'whisper':
    cmd += ['--lyrics-mode', 'whisper']

if SONG_TITLE.strip():
    cmd += ['--title', SONG_TITLE.strip()]

out_name = (SONG_TITLE.strip() or 'song_package') + '.zip'
out_zip = drive_out_dir / out_name
cmd += ['--out', str(out_zip)]

print('🎵 AI 分軌與歌詞處理中，請稍候...')
res = subprocess.run(cmd, check=False)
if res.returncode == 0 and out_zip.exists():
    print('✅ 完成！已自動存入雲端硬碟：麻糬紅豆湯/PitchLab_Songs/' + out_zip.name)
else:
    print('❌ 處理失敗，請查看上方錯誤訊息（若 YouTube 被擋請留空 YOUTUBE_URL 改用上傳本機音檔）。')